In [1]:
import pandas as pd
import numpy as np

MasterDataFrame = pd.read_csv('RawDataFrame.csv')
DropColumn = list(MasterDataFrame.columns)
MasterDataFrame.drop(columns=DropColumn[0],inplace=True)

In [2]:
SplitList = [group for _,group in MasterDataFrame.groupby('countryiso3code')]
#every entry should be of type dataframe now, remove every entry where the number of NaN values = 31 
NoGiniValues = []

for CountryIterator in range(0,len(SplitList)):

    NumberOfNaN = 0
    CurrentCountry = SplitList[CountryIterator]
    GiniColumn = CurrentCountry['SI.POV.GINI']

    for SeriesIterator in GiniColumn:

        if pd.isna(SeriesIterator) == True:
            NumberOfNaN += 1

    if NumberOfNaN == len(GiniColumn):
        NoGiniValues.append(CountryIterator)

NoGiniValues.reverse()

for iterator in NoGiniValues:
    SplitList.pop(iterator)

In [3]:
HalfEmptyIndices = []
ColumnNames = list(SplitList[0].columns) 
ValidEntries = int(0.5 * (14 * 31))

for CountryIterator in range(0,len(SplitList)):
    NumberNaNEntries = 0
    numentries = 0
    CurrentCountry = SplitList[CountryIterator]

    for ColumnIterator in range(3,len(ColumnNames)):
        CurrentColumnName = ColumnNames[ColumnIterator]
        CurrentColumn = CurrentCountry[CurrentColumnName]

        for ValueIterator in CurrentColumn:

            if pd.isna(ValueIterator) == True:
                NumberNaNEntries += 1

    if NumberNaNEntries > ValidEntries:
        HalfEmptyIndices.append(CountryIterator)

HalfEmptyIndices.reverse()

for iterator in HalfEmptyIndices:
    SplitList.pop(iterator)

In [4]:
#make two different interpolation boxes since they need different methods to tackle them 
#to tackle the problems of showing the imputed valuees i just show which vals haven't been imputed
#impute vals for gaps of 1-3 years between two valid values and the  
# Gini loop
for CountryIterator in SplitList:
    OriginalGiniValues = CountryIterator['SI.POV.GINI'].copy()
    NewGiniValues = CountryIterator['SI.POV.GINI'].interpolate(method='linear', limit=3, limit_area='inside')
    CountryIterator['SI.POV.GINI'] = CountryIterator['SI.POV.GINI'].interpolate(method='linear', limit=3, limit_area='inside')
    
    ValuesAreEqual = OriginalGiniValues == NewGiniValues
    BothAreNaN = OriginalGiniValues.isna() & NewGiniValues.isna()
    
    IsOriginal = ValuesAreEqual | BothAreNaN
    
    try:
        CountryIterator.insert(3, 'IsOriginalValue', IsOriginal)
    except ValueError:
        continue
#added the interpolation value

In [5]:
# Features loop
ListOfColumns = list((SplitList[0]).columns)
LengthOfListOfColumns = len(ListOfColumns)

for CountryIterator in SplitList:
    for ColumnIterator in range(4, LengthOfListOfColumns):
        CurrentColumnEntry = ListOfColumns[ColumnIterator]
        
        CurrentColumnBeingImputed = CountryIterator[CurrentColumnEntry].copy()
        
        CurrentColumnBeingImputed = CurrentColumnBeingImputed.ffill(limit=2)
        CurrentColumnBeingImputed = CurrentColumnBeingImputed.interpolate(limit=2)
        CurrentColumnBeingImputed = CurrentColumnBeingImputed.fillna(CurrentColumnBeingImputed.median())
        
        CountryIterator[CurrentColumnEntry] = CurrentColumnBeingImputed


In [ ]:
RegionalClassification = pd.read_excel('CLASS_2025_10_07.xlsx')
RegionalClassification = RegionalClassification.drop(columns=['Economy','Income group','Lending category'])
RegionalClassification = RegionalClassification.rename(columns={'Code':'countryiso3code'})

In [12]:
#now it's time to add all the countries back together 
CleanedMasterDataFrame = pd.concat(SplitList,axis=0)
CleanedMasterDataFrame = pd.merge(CleanedMasterDataFrame,RegionalClassification,how='left',on='countryiso3code')
CleanedMasterDataFrame.to_csv('CleanedMasterDataFrame.csv')